In [ ]:
import os
from os import path
import random
import pickle

import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error

In [ ]:
data_folder = './data'
output_folder = './output'
if not path.exists(output_folder):
    os.makedirs(output_folder)

In [ ]:
with open(path.join(data_folder, 'users.pickle'), 'rb') as f:
    users = pickle.load(f)

## Poisson models for quarantine counts

In [ ]:
df1 = users[['quarantine', 'no_quarantine', 'S1_Q1', 'S1_Q2', 'S1_Q3', 'S1_Q4', 'S1_Q5', 'group']].copy() 
df1.dropna(inplace=True)

# Anchoring variables to minimum (0) so the intercept can be interpreted as the probability of 
# quarantine for a hypothetical particpant who responded 1 to all surveys
df1['S1_Q1'] = df1['S1_Q1'] - 1
df1['S1_Q2'] = df1['S1_Q2'] - 1
df1['S1_Q3'] = df1['S1_Q3'] - 1
df1['S1_Q4'] = df1['S1_Q4'] - 1
df1['S1_Q5'] = df1['S1_Q5'] - 1

df2 = users[['quarantine', 'no_quarantine', 'S2_Q1', 'S2_Q2', 'S2_Q3', 'S2_Q4', 'S1_Q5', 'group']].copy() 
df2.dropna(inplace=True)

df2['S2_Q1'] = df2['S2_Q1'] - 1
df2['S2_Q2'] = df2['S2_Q2'] - 1
df2['S2_Q3'] = df2['S2_Q3'] - 1
df2['S2_Q4'] = df2['S2_Q4'] - 1
df2['S1_Q5'] = df2['S1_Q5'] - 1

In [ ]:
print("\n\n--- H1 model (main effect from group assignment) ---")

formula = 'quarantine ~ group'
poisson_model_h1 = smf.glm(formula=formula, data=df1, family=sm.families.Poisson()).fit()
print(poisson_model_h1.summary())

In [ ]:
print("\n\n--- H1B model (using S2 dataset) ---")

formula = 'quarantine ~ group'
poisson_model_h1b = smf.glm(formula=formula, data=df2, family=sm.families.Poisson()).fit()
print(poisson_model_h1b.summary())

In [ ]:
print("\n\n--- H2 model (interaction of S1 attitudes with group) ---")

formula = 'quarantine ~ (S1_Q1 + S1_Q2 + S1_Q3 + S1_Q4 + S1_Q5) * C(group)'
poisson_model_h2 = smf.glm(formula=formula, data=df1, family=sm.families.Poisson()).fit()
print(poisson_model_h2.summary())

In [ ]:
print("\n\n--- H2B model (using S2 dataset) ---")

formula = 'quarantine ~ (S2_Q1 + S2_Q2 + S2_Q3 + S2_Q4 + S1_Q5) * C(group)'
poisson_model_h2b = smf.glm(formula=formula, data=df2, family=sm.families.Poisson()).fit()
print(poisson_model_h2b.summary())

In [ ]:
with open(path.join(data_folder, 'poisson_model.pickle'), 'wb') as f:
    pickle.dump(poisson_model_h2b, f)

## Health Belief Model

In [ ]:
df_hbm2 = users[['quarantine_rate', 'total_trials', 'S2_Q1', 'S2_Q2', 'S2_Q3', 'S2_Q4', 'S1_Q5', 'group']].copy() 
df_hbm2.dropna(inplace=True)

df_hbm2['S2_Q1'] = df_hbm2['S2_Q1'] - 1
df_hbm2['S2_Q2'] = df_hbm2['S2_Q2'] - 1
df_hbm2['S2_Q3'] = df_hbm2['S2_Q3'] - 1
df_hbm2['S2_Q4'] = df_hbm2['S2_Q4'] - 1
df_hbm2['S1_Q5'] = df_hbm2['S1_Q5'] - 1
df_hbm2['group'] = df_hbm2['group'] - 1

In [ ]:
hbm_formula2 = """
quarantine_rate ~ S2_Q1 + S2_Q2 + S2_Q3 + S2_Q4 + S1_Q5 + group
"""

In [ ]:
# Family: Binomial (for probabilities)
# var_weights: total_trials (tells the model how much weight to give each row)
hbm_model2 = smf.glm(
    formula=hbm_formula2, 
    data=df_hbm2, 
    family=sm.families.Binomial(), 
    var_weights=df_hbm2['total_trials']
).fit()

In [ ]:
print(hbm_model2.summary())

In [ ]:
df_hbm1 = users[['quarantine_rate', 'total_trials', 'S1_Q1', 'S1_Q2', 'S1_Q3', 'S1_Q4', 'S1_Q5', 'group']].copy() 
df_hbm1.dropna(inplace=True)

df_hbm1['S1_Q1'] = df_hbm1['S1_Q1'] - 1
df_hbm1['S1_Q2'] = df_hbm1['S1_Q2'] - 1
df_hbm1['S1_Q3'] = df_hbm1['S1_Q3'] - 1
df_hbm1['S1_Q4'] = df_hbm1['S1_Q4'] - 1
df_hbm1['S1_Q5'] = df_hbm1['S1_Q5'] - 1
df_hbm1['group'] = df_hbm1['group'] - 1

In [ ]:
hbm_formula1 = """
quarantine_rate ~ S1_Q1 + S1_Q2 + S1_Q3 + S1_Q4 + S1_Q5 + group
"""

In [ ]:
# Family: Binomial (for probabilities)
# var_weights: total_trials (tells the model how much weight to give each row)
hbm_model1 = smf.glm(
    formula=hbm_formula1, 
    data=df_hbm1, 
    family=sm.families.Binomial(), 
    var_weights=df_hbm1['total_trials']
).fit()

In [ ]:
print(hbm_model1.summary())

## Predicting in-game attitudes from real-life attitudes

In [ ]:
df_map = users[['S1_Q1', 'S1_Q2', 'S1_Q3', 'S1_Q4', 'S1_Q5', 'S2_Q1', 'S2_Q2', 'S2_Q3', 'S2_Q4']].copy() 
df_map.dropna(inplace=True)
df_map = df_map - 1
len(df_map)

In [ ]:
model_q1 = smf.ols(formula='S2_Q1 ~ S1_Q1 + S1_Q2 + S1_Q3 + S1_Q4 + S1_Q5', data=df_map).fit()
print(model_q1.summary())

In [ ]:
model_q2 = smf.ols(formula='S2_Q2 ~ S1_Q1 + S1_Q2 + S1_Q3 + S1_Q4 + S1_Q5', data=df_map).fit()
print(model_q2.summary())

In [ ]:
model_q3 = smf.ols(formula='S2_Q3 ~ S1_Q1 + S1_Q2 + S1_Q3 + S1_Q4 + S1_Q5', data=df_map).fit()
print(model_q3.summary())

In [ ]:
model_q4 = smf.ols(formula='S2_Q4 ~ S1_Q1 + S1_Q2 + S1_Q3 + S1_Q4 + S1_Q5', data=df_map).fit()
print(model_q4.summary())